# DeepTuneIO Analyzer — Module 10: Access-Pattern Visualization

Notebook for selecting campaigns, JobIDs and visualization profiles. Scientific plotting logic remains in `Module/Access_Pattern_Visualization/`.

**New:** `dxt_timeline` plots MPI-IO and POSIX rank-versus-runtime operations colored by request-size category.


In [ ]:
from pathlib import Path
import sys
from importlib.util import spec_from_file_location, module_from_spec

CODE_ROOT = Path.cwd()  # Carpeta que contiene Module/
MODULE_DIR = (CODE_ROOT / "Module").resolve()
if not MODULE_DIR.is_dir():
    raise FileNotFoundError(f"No se encuentra la carpeta Module: {MODULE_DIR}")
MODULE_FILE = MODULE_DIR / "10_access_pattern_visualization.py"

if not MODULE_FILE.is_file():
    raise FileNotFoundError(f"No se encuentra el módulo: {MODULE_FILE}")

spec = spec_from_file_location("deeptuneio_access_viz10", MODULE_FILE)
module10 = module_from_spec(spec)
sys.modules[spec.name] = module10
spec.loader.exec_module(module10)

print("Module :", MODULE_FILE)
print("Version:", module10.VERSION)


In [ ]:
PROJECT_ROOT = Path(r"D:\DL_Experiments\CORA_DR_UAB").resolve()
ARTICLE_ID = "Tesis"

EVENT = "Tesis"

CAMPAIGNS = {
    "DeepGalaxy": {
        "DG_bw512_f3c5x64": {
            "scenarios": ["lustre_1ost", "lustre_2ost", "lustre_4ost", "nfs_SnGPU_e1"],
            "file_format": ".hdf5",
        },
    },
    "DLIOv1": {
        "HDF5_64bs": {"scenarios": ["lustre_ost"], "file_format": ".h5"},
        "NPZ_64bs": {"scenarios": ["lustre_ost"], "file_format": ".npz"},
        "TFRecord_256kts_64bs": {"scenarios": ["lustre_ost"], "file_format": ".tfrecords"},
        "TFRecord_1mts_64bs": {"scenarios": ["lustre_ost"], "file_format": ".tfrecords"},
    },
}

APPLICATION_POS = 1
DATASET_POS = 1
SCENARIO_POS = 4

APPLICATION = list(CAMPAIGNS)[APPLICATION_POS - 1]
DATASET_GROUP = list(CAMPAIGNS[APPLICATION])[DATASET_POS - 1]
SCENARIO = CAMPAIGNS[APPLICATION][DATASET_GROUP]["scenarios"][SCENARIO_POS - 1]
FILE_FORMAT = CAMPAIGNS[APPLICATION][DATASET_GROUP]["file_format"]

CAMPAIGN_ROOT = (
    PROJECT_ROOT / ARTICLE_ID / "Data"
    / APPLICATION / DATASET_GROUP / SCENARIO
)

print(APPLICATION, DATASET_GROUP, SCENARIO, FILE_FORMAT)
print("Campaign:", CAMPAIGN_ROOT)


## Select the execution dynamically from the experiment matrix

The selector is now **matrix-first**: `experiment_matrix_*.csv` is the source of
truth for configuration metadata, while DXT is used only to confirm that a JobID
has detailed events available.

Normalized modes:
- DeepGalaxy `shared` → `shared`
- DeepGalaxy `shared_reload_shuffle` → `shared_reload_shuffle`
- DLIO `shared` → `shared`
- DLIO `multi` → `file_per_process`


In [ ]:
JOB_TABLE = module10.build_job_selection_table(
    CAMPAIGN_ROOT,
    application=APPLICATION,
    scenario=SCENARIO,
    file_format=FILE_FORMAT,
)

DISPLAY_COLUMNS = [
    "job_id",
    "nodes",
    "process_io",
    "replicate",
    "dxt_available",
    "dxt_events",
]

display(
    JOB_TABLE[
        [c for c in DISPLAY_COLUMNS if c in JOB_TABLE.columns]
    ]
)


In [ ]:
# Optional selector.
TARGET_NODES = None
TARGET_PROCESSES = None
TARGET_MODE = "all"       # shared | shared_reload_shuffle | file_per_process | all
TARGET_FILESYSTEM = "all"
TARGET_OST = None

JOB_CANDIDATES = module10.filter_job_selection_table(
    JOB_TABLE,
    nodes=TARGET_NODES,
    process_io=TARGET_PROCESSES,
    data_loading_mode=TARGET_MODE,
    filesystem=TARGET_FILESYSTEM,
    stripe_count=TARGET_OST,
    dxt_only=True,
)

display(JOB_CANDIDATES[[c for c in DISPLAY_COLUMNS if c in JOB_CANDIDATES.columns]])

if len(JOB_CANDIDATES) == 1:
    JOB_ID = int(JOB_CANDIDATES.iloc[0]["job_id"])
    print("Selected JobID:", JOB_ID)
else:
    JOB_ID = None
    print(f"{len(JOB_CANDIDATES)} DXT candidates found. Refine the filters or choose a JobID from the table.")


In [ ]:
# You may also override the automatic selection manually after inspecting the table:
#JOB_ID = 7296725  # DLIOv1 HDF5 Shared 1N4p L1
#JOB_ID = 7296914 # DLIOv1 HDF5  Shared 16N64p L16
#JOB_ID = 3538214  # DLIOv1 NPZ multi 1N4p L1
#JOB_ID = 3538779 # DLIOv1 NPZ multi 16N64p L16
#JOB_ID = 3528468  # DLIOv1 TFRecord 256K TS multi 1N4p L1
#JOB_ID = 3529616 # DLIOv1 TFRecord 256K TS multi 16N64p L16
#JOB_ID = 3528998  # DLIOv1 TFRecord 1M TS multi 1N4p L1
#JOB_ID = 3529617 # DLIOv1 TFRecord 1M TS multi 16N64p L16


In [ ]:
# Selección manual opcional. None conserva la selección dinámica de la tabla.
MANUAL_JOB_ID = None  # Ejemplo: 5333326 (solo si pertenece a esta campaña)
if MANUAL_JOB_ID is not None:
    available_ids = set(JOB_TABLE["job_id"].dropna().astype(str))
    if str(MANUAL_JOB_ID) not in available_ids:
        raise ValueError(f"JobID {MANUAL_JOB_ID} no figura en la campaña seleccionada")
    JOB_ID = int(MANUAL_JOB_ID)
print("JobID seleccionado:", JOB_ID)


In [ ]:
# ============================================================
# Access-pattern visualization profile
# ============================================================
#
# Select which family of DXT access-pattern figures to generate.
#
# Available options:
#
# "process3d"
#     Process-oriented spatial/temporal analysis.
#     Generates:
#       01_spatial_temporal_3d
#          X = I/O Process
#          Y = Operation Order
#          Z = File Offset
#          Color/size = Request Size
#
#       04_spatial_pattern_by_process
#          X = I/O Process
#          Y = File Offset
#          Color/size = Request Size
#
# "temporal2d"
#     Classical 2D spatial and temporal analysis.
#     Generates:
#       02_spatial_pattern
#          X = Operation Order
#          Y = File Offset
#          Color/size = Request Size
#
#       03_temporal_pattern
#          X = Start Time
#          Y = File Offset
#          Color/size = Request Size
#
# "temporal3d"
#     Real-time process-aware spatio-temporal analysis.
#     Generates:
#       07_spatial_temporal_time_3d
#          X = I/O Process
#          Y = Start Time
#          Z = File Offset
#          Color/size = Request Size
#
#       03_temporal_pattern
#          X = Start Time
#          Y = File Offset
#          Color/size = Request Size
#
# "hist"
#     Spatial access pattern with marginal histograms.
#     Generates:
#       05_spatial_pattern_histograms
#
# "ost"
#     DXT-reported OST association overview.
#     Generates:
#       06_ost_association_overview
#
# "dxt_timeline"
#     DXT rank-versus-runtime timeline (MPI-IO and POSIX panels).
#     Each operation is a segment from start to end time; color encodes request size.
#     Outputs 08_dxt_rank_timeline.png and .pdf.
#     Interfaces are only separated when the source records identify them.
#
# "all"
#     Generates all available access-pattern figure families:
#       01, 02, 03, 04, 05, 06 and 07.
#
# Examples:
#     PROFILE = "process3d"
#     PROFILE = "temporal2d"
#     PROFILE = "temporal3d"
#     PROFILE = "hist"
#     PROFILE = "ost"
#     PROFILE = "dxt_timeline"
#     PROFILE = "all"
#
# ============================================================

PROFILE = "dxt_timeline"  # Other options: process3d, temporal2d, temporal3d, hist, ost, all

In [ ]:
# You may also override the automatic selection manually after inspecting the table:
# JOB_ID = 4935078

# all | read | write
OPERATION = "read"

# Orientación 3D: solo configurable cuando el módulo Python
# admita --view-elev y --view-azim.
VIEW_ELEV = 15
VIEW_AZIM = -30


# Optional deterministic visual downsampling for very dense traces.
# Keep None for exact full-trace figures.
MAX_EVENTS = None

print("JobID:", JOB_ID)


In [ ]:
# El módulo Python actual no admite --view-elev ni --view-azim.
# No enviar argumentos que argparse no reconoce.
args = [
    "--campaign-root", str(CAMPAIGN_ROOT),
    "--application", APPLICATION,
    "--dataset-group", DATASET_GROUP,
    "--scenario", SCENARIO,
    "--file-format", FILE_FORMAT,
    "--event", EVENT,
    "--operation", OPERATION,
    "--profile", PROFILE,
]

if JOB_ID is not None:
    args += ["--job-id", str(JOB_ID)]
if MAX_EVENTS is not None:
    args += ["--max-events", str(MAX_EVENTS)]

print("Ejecutando módulo 10:", args)
return_code = module10.main(args)
print("Return code:", return_code)
if return_code != 0:
    raise RuntimeError(f"Module 10 failed with return code {return_code}")


## Article 01 — Fig. 10(b,d) spatial model

Version 1.1 adds the missing 2D spatial representation:

- **X:** I/O Process
- **Y:** File Offset
- **Color:** Read Request Size

A 4-process JobID reproduces the Fig. 10(b) model; a 64-process JobID reproduces the Fig. 10(d) model.

In [ ]:
OUTPUT_ROOT = (
    CAMPAIGN_ROOT / "results" / "Figures" / "Access_Pattern"
    / (f"job_{JOB_ID}" if JOB_ID is not None else "campaign")
)

print("Outputs:", OUTPUT_ROOT)

if OUTPUT_ROOT.exists():
    for p in sorted(OUTPUT_ROOT.iterdir()):
        print(" -", p.name)


## DXT rank timeline — notebook validation

Run the cells above with `PROFILE = "dxt_timeline"`. The figure uses the currently selected campaign, JobID and operation.

**Checks:** inspect generated files, verify PNG/PDF exist, and compare operation counts and interface labels with the original DXT source. A successful plot is not sufficient to validate scientific equivalence.


In [ ]:
from pathlib import Path
from IPython.display import display, Image

# Validation/preview for the selected JobID.
expected = [OUTPUT_ROOT / "08_dxt_rank_timeline.png", OUTPUT_ROOT / "08_dxt_rank_timeline.pdf"]
for artifact in expected:
    print(("OK" if artifact.is_file() else "MISSING"), artifact)
if expected[0].is_file():
    display(Image(filename=str(expected[0])))
else:
    print("Check the profile output, source availability and the generated manifest.")


In [ ]:
from pathlib import Path
import pandas as pd

analysis_dir = (
    CAMPAIGN_ROOT
    / "results"
    / "Access_Pattern_DXT"
    / "Analysis"
)

traces_dir = (
    CAMPAIGN_ROOT
    / "results"
    / "Access_Pattern_DXT"
    / "Traces"
)

SOURCE_COMPARISON_JOB_ID = JOB_ID  # Use currently selected job

analysis_files = sorted(
    analysis_dir.glob(f"*{SOURCE_COMPARISON_JOB_ID}*_dxt_analysis.csv")
)

trace_files = sorted(
    traces_dir.glob(f"*{SOURCE_COMPARISON_JOB_ID}*_dxt.csv")
)

print("=" * 80)
print("DXT SOURCE COMPARISON")
print("=" * 80)

print("\nAnalysis files:")
for p in analysis_files:
    df = pd.read_csv(p)
    print(f"  {p.name}")
    print(f"    rows    : {len(df)}")
    print(f"    columns : {len(df.columns)}")

print("\nTrace files:")
for p in trace_files:
    df = pd.read_csv(p)
    print(f"  {p.name}")
    print(f"    rows    : {len(df)}")
    print(f"    columns : {len(df.columns)}")

if analysis_files and trace_files:

    a = pd.read_csv(analysis_files[0])
    t = pd.read_csv(trace_files[0])

    print("\n" + "=" * 80)
    print("STRUCTURAL COMPARISON")
    print("=" * 80)

    print("Analysis rows :", len(a))
    print("Trace rows    :", len(t))

    common = sorted(
        set(a.columns).intersection(t.columns)
    )

    print("Common columns:", len(common))

    print("\nAnalysis-only columns:")
    print(sorted(set(a.columns) - set(t.columns)))

    print("\nTrace-only columns:")
    print(sorted(set(t.columns) - set(a.columns)))

    print("\nCommon columns:")
    print(common)